# 🔍 Deepfake Detection — SWIN Transformer Training

This notebook fine-tunes a **SWIN-Tiny Transformer** on the [Hemg/deepfake-and-real-images](https://huggingface.co/datasets/Hemg/deepfake-and-real-images) dataset for binary deepfake classification (Real vs Fake).

**Instructions:**
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run all cells in order
3. When prompted, enter your Hugging Face token
4. The trained model will be uploaded to your Hugging Face account automatically

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate accelerate huggingface_hub scikit-learn pillow torch torchvision

## 2. Login to Hugging Face

Get your token from: https://huggingface.co/settings/tokens  
Create a **Write** token if you don't have one.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Hemg/deepfake-and-real-images")
print(dataset)
print(f"\nLabels: {dataset['train'].features['label'].names}")
print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

## 4. Setup Model & Preprocessing

In [ ]:
import torch
import numpy as np
import evaluate
from transformers import AutoImageProcessor, SwinForImageClassification, TrainingArguments, Trainer

MODEL_NAME = "microsoft/swin-tiny-patch4-window7-224"

# Your HuggingFace username — UPDATE THIS
HF_USERNAME = "Purnachander-Konda"
HF_REPO_NAME = f"{HF_USERNAME}/deepfake-detection-swin"

labels = dataset['train'].features['label'].names
num_labels = len(labels)
id2label = {str(i): label for i, label in enumerate(labels)}
label2id = {label: str(i) for i, label in enumerate(labels)}

print(f"Number of classes: {num_labels}")
print(f"Labels: {id2label}")

# Load processor and model
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = SwinForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print(f"\n✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def preprocess(batch):
    """Preprocess images for SWIN Transformer."""
    images = [img.convert('RGB') for img in batch['image']]
    inputs = processor(images=images, return_tensors='pt')
    inputs['label'] = batch['label']
    return inputs

def collate_fn(batch):
    """Collate function for DataLoader."""
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['label'] for x in batch])
    }

# Apply preprocessing
processed_train = dataset['train'].with_transform(preprocess)
processed_test = dataset['test'].with_transform(preprocess)

print(f"✓ Preprocessing pipeline ready")

## 5. Setup Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(pred):
    """Compute accuracy, F1, precision, and recall."""
    preds = np.argmax(pred.predictions, axis=1)
    refs = pred.label_ids
    
    results = {}
    results.update(accuracy_metric.compute(predictions=preds, references=refs))
    results.update(f1_metric.compute(predictions=preds, references=refs, average='macro'))
    results.update(precision_metric.compute(predictions=preds, references=refs, average='macro'))
    results.update(recall_metric.compute(predictions=preds, references=refs, average='macro'))
    return results

print("✓ Metrics ready: Accuracy, F1, Precision, Recall")

## 6. Train the Model

This takes approximately **30-60 minutes** on a T4 GPU.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    remove_unused_columns=False,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    num_train_epochs=3,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
    hub_model_id=HF_REPO_NAME,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    train_dataset=processed_train,
    eval_dataset=processed_test,
    tokenizer=processor,
)

print("✓ Trainer configured. Starting training...\n")
train_results = trainer.train()
print("\n🎉 Training complete!")

## 7. Evaluate & Print Results

In [ ]:
metrics = trainer.evaluate(processed_test)

print("\n" + "="*50)
print("📊 FINAL TEST RESULTS")
print("="*50)
print(f"  Accuracy:  {metrics.get('eval_accuracy', 0):.4f}")
print(f"  F1 Score:  {metrics.get('eval_f1', 0):.4f}")
print(f"  Precision: {metrics.get('eval_precision', 0):.4f}")
print(f"  Recall:    {metrics.get('eval_recall', 0):.4f}")
print("="*50)
print("\n⬆️ Copy these values into your README.md Results section!")

## 8. Upload Model to Hugging Face Hub

In [ ]:
trainer.push_to_hub(commit_message="Upload trained SWIN-Tiny deepfake detection model")

print(f"\n🚀 Model uploaded to: https://huggingface.co/{HF_REPO_NAME}")
print(f"\nNext step: Deploy on Hugging Face Spaces!")

## 9. Quick Test

In [ ]:
from transformers import pipeline

# Test the uploaded model
pipe = pipeline("image-classification", model=HF_REPO_NAME)

# Test on a sample from the test set
sample = dataset['test'][0]
result = pipe(sample['image'])

print(f"True label: {labels[sample['label']]}")
print(f"Predictions:")
for r in result:
    print(f"  {r['label']}: {r['score']:.4f}")

print(f"\n✅ All done! Your model is live at:")
print(f"   https://huggingface.co/{HF_REPO_NAME}")